# Data Profiling: E-commerce Source Data
Checking nulls, cardinality, and basic distributions for the raw staging tables.

In [1]:
import duckdb
import pandas as pd

conn = duckdb.connect('../data/ecommerce_dwh.duckdb')

In [2]:
# Check record counts
conn.execute("""
    SELECT 'stg_orders' as table_name, COUNT(*) as record_count FROM stg_orders
    UNION ALL
    SELECT 'stg_customers', COUNT(*) FROM stg_customers
    UNION ALL
    SELECT 'stg_products', COUNT(*) FROM stg_products
""").df()

,table_name,record_count
0,stg_orders,1200000
1,stg_customers,12024
2,stg_products,5000


In [3]:
# Order status distribution
conn.execute("""
    SELECT status, COUNT(*) as cnt, 
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) as pct
    FROM stg_orders
    GROUP BY status
    ORDER BY cnt DESC
""").df()

,status,cnt,pct
0,Delivered,840161,70.01
1,Shipped,179826,14.99
2,Returned,60397,5.03
3,Cancelled,60038,5.00
4,Processing,59578,4.96


In [4]:
# Null checks on customers
conn.execute("""
    SELECT 
        COUNT(*) as total_rows,
        SUM(CASE WHEN city IS NULL THEN 1 ELSE 0 END) as null_cities,
        SUM(CASE WHEN state IS NULL THEN 1 ELSE 0 END) as null_states
    FROM stg_customers
""").df()

,total_rows,null_cities,null_states
0,12024,0.0,0.0
